# 실습: OpenAI API 기본 호출

OpenAI Python SDK와 Responses API를 사용해 첫 호출부터 응답 구조 분석까지 진행합니다.


## 수업 진행 방식

- API 요청이 발생하는 셀은 한 번 실행한 뒤 결과를 확인합니다.
- 같은 셀을 여러 번 실행하면 비용이 누적될 수 있습니다.
- 에러가 발생하면 에러 메시지를 먼저 읽고 원인을 적어봅니다.
- 모든 API 키와 모델명은 `.env` 파일에서 읽습니다.


## 0. 준비 확인


In [1]:
import os
from pprint import pprint

import tiktoken
from dotenv import load_dotenv
from openai import (
    APIError,
    AuthenticationError,
    BadRequestError,
    OpenAI,
    RateLimitError,
)

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 먼저 설정하세요.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI(api_key=api_key)

print(f"사용 모델: {MODEL}")


사용 모델: gpt-5.4-mini


## 1. 공통 도우미 함수


In [2]:
def print_usage(response) -> None:
    usage = getattr(response, "usage", None)
    if usage is None:
        print("토큰 사용량 정보가 없습니다.")
        return

    print("\n[토큰 사용량]")
    print(f"입력 토큰: {getattr(usage, 'input_tokens', None)}")
    print(f"출력 토큰: {getattr(usage, 'output_tokens', None)}")
    print(f"총 토큰:   {getattr(usage, 'total_tokens', None)}")


def call_openai(label, request_func):
    print(f"\n=== {label} ===")
    try:
        response = request_func()
    except AuthenticationError as exc:
        print("AuthenticationError: API 키가 없거나 올바르지 않습니다.")
        print(exc)
        return None
    except RateLimitError as exc:
        print("RateLimitError: 요청 한도 또는 사용량 한도를 초과했습니다.")
        print(exc)
        return None
    except BadRequestError as exc:
        print("BadRequestError: 요청 형식, 모델명, 파라미터를 확인하세요.")
        print(getattr(exc, "message", str(exc)))
        return None
    except APIError as exc:
        print("APIError: OpenAI API 호출 중 오류가 발생했습니다.")
        print(exc)
        return None
    except Exception as exc:
        print("예상치 못한 오류가 발생했습니다.")
        print(type(exc).__name__, exc)
        return None

    text = getattr(response, "output_text", None)
    if text:
        print(text)

    print_usage(response)
    return response


## 2. 첫 번째 OpenAI API 호출


In [3]:
response = call_openai(
    "첫 번째 OpenAI API 호출",
    lambda: client.responses.create(
        model=MODEL,
        input="Python을 처음 배우는 학생에게 변수의 의미를 한 문장으로 설명해줘.",
        max_output_tokens=180,
    ),
)



=== 첫 번째 OpenAI API 호출 ===
파이썬에서 **변수**는 **값을 저장해 두고 필요할 때 꺼내 쓰는 이름표**입니다.

[토큰 사용량]
입력 토큰: 25
출력 토큰: 34
총 토큰:   59


## 3. 응답 구조 분석


In [4]:
response = call_openai(
    "응답 구조 분석용 호출",
    lambda: client.responses.create(
        model=MODEL,
        input="딕셔너리를 초보자에게 설명하고, 짧은 예시 코드도 보여줘.",
        max_output_tokens=250,
    ),
)

if response is not None:
    print("\n[기본 필드]")
    print("id:", response.id)
    print("model:", response.model)
    print("status:", response.status)

    print("\n[output 항목 타입]")
    for index, item in enumerate(response.output, start=1):
        print(index, getattr(item, "type", None))

    print("\n[usage 원본]")
    pprint(response.usage.model_dump() if hasattr(response.usage, "model_dump") else response.usage)



=== 응답 구조 분석용 호출 ===
물론이죠.  
**딕셔너리(dictionary)**는 파이썬에서 **“키(key)와 값(value)을 짝지어 저장하는 자료형”**입니다.

## 쉽게 말하면
- **키**: 값을 찾기 위한 이름
- **값**: 실제 저장된 데이터

예를 들어,
- `"이름"` → `"철수"`
- `"나이"` → `20`
- `"도시"` → `"서울"`

이렇게 저장해두면, 숫자 대신 **이름표(키)**로 원하는 값을 바로 찾을 수 있습니다.

## 특징
- **중괄호 `{}`**를 사용합니다.
- 데이터는 **키: 값** 형태로 들어갑니다.
- **키는 중복될 수 없고**, 값은 중복될 수 있습니다.
- 순서를 완전히 기준으로 쓰기보다는, **키로 빠르게 찾는 것**이 핵심입니다.

## 짧은 예시 코드
```python
person = {
    "name": "철수",
    "age": 20,
    "city": "서울"
}

print(person["name"])  # 철수


[토큰 사용량]
입력 토큰: 28
출력 토큰: 250
총 토큰:   278

[기본 필드]
id: resp_0ed1fad1c9d699de006a471b33782c8199a32fcd3a6b1b1251
model: gpt-5.4-mini-2026-03-17
status: incomplete

[output 항목 타입]
1 message

[usage 원본]
{'input_tokens': 28,
 'input_tokens_details': {'cached_tokens': 0},
 'output_tokens': 250,
 'output_tokens_details': {'reasoning_tokens': 0},
 'total_tokens': 278}


## 4. 지시문 사용


In [5]:
role = "친절한 한국어 Python 선생님"

response = call_openai(
    "instructions로 역할 지정",
    lambda: client.responses.create(
        model=MODEL,
        instructions=(
            f"당신은 {role}입니다. "
            "답변은 쉬운 비유 1개, 코드 1개, 확인 질문 1개로 구성하세요."
        ),
        input="리스트가 무엇인지 설명해줘.",
        max_output_tokens=350,
    ),
)



=== instructions로 역할 지정 ===
리스트는 **여러 값을 한 상자에 순서대로 담아두는 것**이에요.  
비유하자면, **도서관 책장**처럼 생각하면 됩니다. 책장에는 책이 여러 권 들어가고, **몇 번째 칸에 어떤 책이 있는지 순서가 정해져** 있죠. 파이썬의 리스트도 마찬가지로 값들을 순서대로 저장합니다.

```python
fruits = ["사과", "바나나", "딸기"]
print(fruits)      # 전체 리스트
print(fruits[0])   # 첫 번째 값
```

원하시면 제가 다음으로 **리스트를 만드는 방법, 추가/삭제하는 방법**도 이어서 설명해드릴까요?

[토큰 사용량]
입력 토큰: 58
출력 토큰: 168
총 토큰:   226


### 역할 바꿔보기


In [6]:
role_examples = [
    "10살 학생에게 설명하는 선생님",
    "면접관처럼 핵심만 묻고 답하는 코치",
    "실무 개발자처럼 주의점을 강조하는 멘토",
]

for role in role_examples:
    call_openai(
        f"역할: {role}",
        lambda role=role: client.responses.create(
            model=MODEL,
            instructions=f"당신은 {role}입니다. 답변은 5문장 이내로 작성하세요.",
            input="파이썬 함수가 왜 필요한지 설명해줘.",
            max_output_tokens=220,
        ),
    )



=== 역할: 10살 학생에게 설명하는 선생님 ===
파이썬 함수는 자주 쓰는 일을 한 번 정해두고, 필요할 때마다 다시 꺼내 쓰는 도구예요.  
예를 들어 “두 숫자를 더하기” 같은 일을 함수로 만들면, 매번 같은 코드를 다시 쓰지 않아도 돼요.  
그래서 코드를 더 짧고, 읽기 쉽고, 실수도 줄일 수 있어요.  
또 한 번 만든 함수는 여러 곳에서 재사용할 수 있어서 아주 편리해요.

[토큰 사용량]
입력 토큰: 51
출력 토큰: 114
총 토큰:   165

=== 역할: 면접관처럼 핵심만 묻고 답하는 코치 ===
파이썬 함수는 **코드를 재사용**하게 해줘서 같은 일을 여러 번 쓸 때 매번 다시 작성하지 않아도 됩니다.  
또한 코드를 **작은 단위로 나눠서** 읽고 이해하기 쉽게 만들고, **유지보수**도 쉬워집니다.  
입력값을 받아 결과를 내는 형태로 만들면, 복잡한 로직도 **명확하게 구조화**할 수 있습니다.  
즉, 함수는 **중복을 줄이고, 가독성을 높이고, 수정과 테스트를 쉽게** 하기 위해 필요합니다.

[토큰 사용량]
입력 토큰: 55
출력 토큰: 134
총 토큰:   189

=== 역할: 실무 개발자처럼 주의점을 강조하는 멘토 ===
파이썬 함수는 **자주 쓰는 코드를 묶어서 재사용**하게 해줘서, 같은 일을 여러 번 쓸 때 코드 중복을 줄여줍니다.  
또한 코드를 기능별로 나눠서 **읽기 쉽고 관리하기 쉽게** 만들어 줍니다.  
예를 들어 입력 처리, 계산, 출력처럼 역할을 나누면 버그를 찾거나 수정할 때 훨씬 편합니다.  
실무에서는 함수가 없으면 코드가 금방 길어지고 복잡해져서, 협업과 유지보수가 매우 어려워집니다.  
즉, 함수는 **재사용성, 가독성, 유지보수성**을 높이기 위해 필요합니다.

[토큰 사용량]
입력 토큰: 54
출력 토큰: 162
총 토큰:   216


## 5. 토큰 사전 추정


In [7]:
texts = [
    "안녕하세요.",
    "Python에서 리스트와 튜플의 차이를 설명해줘.",
    "초보자를 위한 함수 개념 설명과 예시 코드 2개를 작성해줘.",
]

try:
    encoding = tiktoken.encoding_for_model(MODEL)
except KeyError:
    encoding = tiktoken.get_encoding("o200k_base")

for text in texts:
    token_count = len(encoding.encode(text))
    print(f"{token_count:>4} tokens | {text}")


   3 tokens | 안녕하세요.
  14 tokens | Python에서 리스트와 튜플의 차이를 설명해줘.
  20 tokens | 초보자를 위한 함수 개념 설명과 예시 코드 2개를 작성해줘.


## 6. 실제 사용량과 비교


In [8]:
prompt = "토큰을 아껴야 하는 이유를 Python API 수업 관점에서 설명해줘."
estimated_tokens = len(encoding.encode(prompt))
print(f"호출 전 입력 토큰 추정: {estimated_tokens}")

response = call_openai(
    "토큰 사용량 비교",
    lambda: client.responses.create(
        model=MODEL,
        input=prompt,
        max_output_tokens=180,
    ),
)


호출 전 입력 토큰 추정: 20

=== 토큰 사용량 비교 ===
Python API 수업 관점에서 **토큰을 아껴야 하는 이유**는 단순히 “비용 절감”만이 아니라, **실습 효율, 응답 품질, 운영 안정성**과도 연결되기 때문입니다.

## 1. API 호출 비용이 직접 줄어듭니다
Python으로 OpenAI 같은 API를 사용할 때는 보통 **입력 토큰 + 출력 토큰** 기준으로 비용이 계산됩니다.  
즉, 같은 기능을 해도:

- 불필요하게 긴 프롬프트
- 지나치게 긴 대화 기록
- 중복된 설명

이 많으면 **API 사용료가 더 많이 발생**합니다.

예를 들어 수업에서 실습을 많이 하게 되면, 작은 차이도 누적되어 비용 차이가 꽤 커집

[토큰 사용량]
입력 토큰: 26
출력 토큰: 180
총 토큰:   206


## 7. 에러 처리


In [9]:
# 일부러 잘못된 API 키를 사용해 인증 오류를 관찰합니다.
# 실제 수업에서는 이 셀을 실행한 뒤 에러 메시지를 읽고 다음 셀로 넘어갑니다.

bad_client = OpenAI(api_key="sk-invalid-key-for-class-demo")

bad_response = call_openai(
    "잘못된 API 키 테스트",
    lambda: bad_client.responses.create(
        model=MODEL,
        input="이 요청은 실패해야 정상입니다.",
    ),
)



=== 잘못된 API 키 테스트 ===
AuthenticationError: API 키가 없거나 올바르지 않습니다.
Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-inval*****************demo. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


### 자주 만나는 오류

| 오류 | 확인할 것 |
| --- | --- |
| AuthenticationError | `.env`의 `OPENAI_API_KEY` 값 |
| RateLimitError | 짧은 시간에 너무 많이 호출했는지 |
| BadRequestError | 모델명, 파라미터, 입력 형식 |
| 결제 관련 오류 | OpenAI 프로젝트의 Billing 설정 |


## 8. 사용 가능한 모델 목록 조회


In [10]:
models_response = call_openai(
    "모델 목록 조회",
    lambda: client.models.list(),
)

if models_response is not None:
    for model in models_response.data[:20]:
        print(model.id)



=== 모델 목록 조회 ===
토큰 사용량 정보가 없습니다.
text-embedding-ada-002
whisper-1
gpt-3.5-turbo
tts-1
gpt-3.5-turbo-16k
gpt-4-0613
gpt-4
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
